In [1]:
# Imports
import random
import json
import pickle
import numpy as np
import nltk

In [2]:

# Installing nltk libraries needed for this script to function
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Error loading punkt: <urlopen error [Errno 11001]
[nltk_data]     getaddrinfo failed>
[nltk_data] Error loading punkt_tab: <urlopen error [Errno 11001]
[nltk_data]     getaddrinfo failed>
[nltk_data] Error loading wordnet: <urlopen error [Errno 11001]
[nltk_data]     getaddrinfo failed>
[nltk_data] Error loading omw-1.4: <urlopen error [Errno 11001]
[nltk_data]     getaddrinfo failed>


False

In [3]:
# more imports
from nltk.stem import WordNetLemmatizer
from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
from keras.optimizers import SGD

c:\Users\mrtom\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.3.0) or chardet (7.1.0)/charset_normalizer (3.4.3) doesn't match a supported version!
  warnings.warn(


In [4]:
# Lematizer returns the shortest lemma found in WordNet, or the input string unchanged if nothing is found.
# For our case-study, that will be the simplest term from our patterns lists, grouping 'em in similar ideas, so stuff like walk, walking, walked, will all carry similarities and seen as related terms.
lematizer = WordNetLemmatizer()

In [5]:
# Loading the intents json file containing all the original intents I've hard-coded.
intents = json.loads(open("../intents.json").read())

In [6]:
words = []  # This will hold the actual words found in the patterns, we tokenize them.
classes = []  # This will hold the tags to those tokenized words.
documents = []  # This will hold all tokenized words and their tags as tuples.
ignore_letters = [
    "?",
    "!",
    ",",
    ".",
]  # These characters would not be needed in the tokenization

In [7]:
for intent in intents["intents"]:

    for pattern in intent["patterns"]:
        # Tokenizing the words in each pattern,and adding them to our words list
        word_list = nltk.word_tokenize(pattern)
        words.extend(word_list)

        # Adding each tokenized word with their respective tag as a tuple
        documents.append((word_list, intent["tag"]))

        # Adding each new tag into the known tags
        if intent["tag"] not in classes:
            classes.append(intent["tag"])

In [8]:
# Lemmatizing the tokenized words and then sorting them.
words = [lematizer.lemmatize(word) for word in words if word not in ignore_letters]
words = sorted(set(words))

# Not really needed, but it's a good practice.
classes = sorted(set(classes))


# We then save the words and classes to our directory.
pickle.dump(words, open("../training_outputs/words.pkl", "wb"))
pickle.dump(classes, open("../training_outputs/classes.pkl", "wb"))

Neural nets struggle working with words as 'strings' we'd have to categorize them as numbers.

here's the game plan:
 Each "pattern" has a unique tag. So if I had a list for each pattern holding every know tag from our intents.json. Encoding each pattern's tag to then become 1 whilst the others become 0. That way the Neural nets' not working blind anymore.

*This approach is known as a bag of words.*

In [9]:

training = [] #An empty list that would hold a neural net ready version of the tuples in documents.
# A list filled with as many 0s as there are classes. Serves as a base for the tags hot keying.
output_empty = [0] * len(classes)

for document in documents:
    # Remember each document is a tuple holding tokenized words and then the tag.
    bag = [] #We start with an empty bag
    
    print(f"Considering document: {document}")

    word_patterns = document[0] #The word pattern here is the tokenized terms.
    
    word_patterns = [lematizer.lemmatize(word.lower()) for word in word_patterns] #Lemmatizing each word in the tokenized words.
    for word in words:
        bag.append(1) if word in word_patterns else bag.append(0) # NB: 1 and 0 here's not for indexing. So for everry word that we know of: If the lemmatized word already exists, 1. If it doesn't 0. Voila!! Bag of words.

    output_row = list(output_empty) #output row would be our finalized bag of words, specifying which tag is 0 or 1.
    
    tag = document[1]
    # 'classes' holds the tags remember, so we find the index of the tag of our document there, and flip that value from 0 to 1.
    output_row[classes.index(tag)] = 1
    training.append([bag, output_row]) # training now holds the neural net ready version of the document   

Considering document: (['hello'], 'greetings')
Considering document: (['hey'], 'greetings')
Considering document: (['hi'], 'greetings')
Considering document: (['good', 'day'], 'greetings')
Considering document: (['Greetings'], 'greetings')
Considering document: (['What', "'s", 'up', '?'], 'greetings')
Considering document: (['how', 'is', 'it', 'going', '?'], 'greetings')
Considering document: (['cya'], 'goodbye')
Considering document: (['See', 'ya', 'later'], 'goodbye')
Considering document: (['Goodbye'], 'goodbye')
Considering document: (['I', 'am', 'leaving'], 'goodbye')
Considering document: (['Have', 'a', 'good', 'day'], 'goodbye')
Considering document: (['bye'], 'goodbye')
Considering document: (['cao'], 'goodbye')
Considering document: (['see', 'ya'], 'goodbye')
Considering document: (['how', 'old'], 'age')
Considering document: (['how', 'old', 'is', 'florian'], 'age')
Considering document: (['what', 'is', 'your', 'age'], 'age')
Considering document: (['How', 'old', 'are', 'you']

In [10]:
random.shuffle(training) # We don't need to see any patterns based on positioning, shuffling the training parameters.
training = np.array(training, dtype = object)

train_x = list(training[:, 0])
train_y = list(training[:, 1])


model = Sequential()
model.add(Dense(128, input_shape=(len(train_x[0]),), activation="relu"))
model.add(Dropout(0.5))
model.add(Dense(64, activation="relu"))
model.add(Dropout(0.5))
model.add(Dense(len(train_y[0]), activation="softmax"))


sgd = SGD(learning_rate= 0.01, decay = 1e-6, momentum=0.9, nesterov = True)
model.compile(loss="categorical_crossentropy", optimizer=sgd, metrics=["accuracy"])

hist = model.fit(np.array(train_x), np.array(train_y), epochs = 200, batch_size=5, verbose = 1)
model.save("../model/chatbot_model.h5", hist)
print("Done")

c:\Users\mrtom\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\mrtom\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


Epoch 1/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.1714 - loss: 2.0157   
Epoch 2/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.2571 - loss: 1.9116 
Epoch 3/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.3143 - loss: 1.9087
Epoch 4/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.2000 - loss: 1.8606     
Epoch 5/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.3143 - loss: 1.7729
Epoch 6/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.2571 - loss: 1.8238 
Epoch 7/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.2857 - loss: 1.7636 
Epoch 8/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.4857 - loss: 1.6718 
Epoch 9/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.4000 - loss: 1.6602 
Epoch 10/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5429 - loss: 1.5652
Epoch 11/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.4857 - loss: 1.5201 
Epoch 12/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.628

Done
